<h1 align = "Center">
Lotus Group Customer Behavior Analysis

<h2 align = "Center">
Data Cleaning and Integration
</h1>

In [1]:

import pandas as pd
from pathlib import Path

# Define path to folder
data_path = Path(r"../01_data/raw")

def load_csv (filename):
    return pd.read_csv(data_path/filename)

# Customer Records Cleaning

We want to establish a reliable customer dimension. We already identified:

- 3,050 customer records.

- 50 duplicated records, representing 50 customers appearing twice.

- 3,000 distinct customer IDs.

- 567 invalid birth dates.

- 382 missing email addresses.

- Inconsistent capitalization in the gender column.

We start by creatinga  working copy

In [2]:
# Preserve the original dataset and create a working copy
customers = load_csv("tbl_customers.csv")
customers_clean = customers.copy()

print("Original records:", len(customers))
print("Working copy records:", len(customers_clean))

Original records: 3050
Working copy records: 3050


We've established earlier that the 50 duplicated customer records are exact duplicates, not conflicting customer profiles.

Therefore, retaining one copy of each record is appropriate.

What we expect:
- Records after removing duplicates: 3,000.
- Remaining duplicate customer IDs: 0.

Why are we doing this?

We expect our analysis to eventually calculate customer-level metrics such as:
- Total historical spending.
- Number of purchases.
- Average order value.
- Purchase frequency.
- Customer tenure.

If a customer appears more than once in the customer dimension, joining that dimension to transaction records could duplicate transactions and inflate these metrics.

Removing exact duplicates prevents that problem.

In [3]:
# Remove exact duplicate records
customers_clean = customers_clean.drop_duplicates()

print("Records after removing duplicates:", len(customers_clean))

print(
    "Remaining duplicate customer IDs:",
    customers_clean["customer_id"].duplicated().sum()
)

Records after removing duplicates: 3000
Remaining duplicate customer IDs: 0


## Standardizing Customer Gender

We've identified six variations:

`Male`, `male`, `MALE`, `Female`, `female`, `FEMALE`

These represent only two actual categories. We'll standardize capitalization to avoid treating them as separate categories during analysis.

**Why it matters**

When we eventually investigate customer purchasing behaviour, we may want to compare spending and purchasing frequency across demographic groups.

Standardization ensures that all customers belonging to the same category are grouped correctly.

In [4]:
# Standardize gender capitalization and remove surrounding whitespace
customers_clean["gender"] = (
    customers_clean["gender"]
    .str.strip()
    .str.capitalize()
)

# Verify the results
print(customers_clean["gender"].value_counts(dropna=False))

gender
Male      1669
Female    1331
Name: count, dtype: int64


## Handling invalid Birth Dates

We previously discovered 567 invalid birth dates.

We want to convert the column into a proper datetime field. Invalid entries will automatically become missing values.

In [5]:
# Convert birth_date to datetime
customers_clean["birth_date"] = pd.to_datetime(
    customers_clean["birth_date"],
    errors="coerce"
)

# Check the results
print("Birth date data type:", customers_clean["birth_date"].dtype)

print(
    "Missing/invalid birth dates:",
    customers_clean["birth_date"].isna().sum()
)

Birth date data type: datetime64[us]
Missing/invalid birth dates: 560


We are deliberately not imputing these missing dates because there is no reliable basis for reconstructing a customer's date of birth.

## Convert Registration Dates and Phone Numbers

We want to convert `registration_date` to datetime so we can calculate customer tenure later.

We also want to convert `phone` to text because phone numbers are identifiers, not quantities. We should not perform mathematical operations on them.

In [6]:
# Convert registration_date to datetime
customers_clean["registration_date"] = pd.to_datetime(
    customers_clean["registration_date"],
    errors="coerce"
)

# Convert phone numbers to strings
customers_clean["phone"] = (
    customers_clean["phone"]
    .astype("string")
)

# Verify data types
print(customers_clean[["registration_date", "phone"]].dtypes)

# Check for missing values
print("\nMissing values:")
print(
    customers_clean[
        ["registration_date", "phone", "email"]
    ].isna().sum()
)

registration_date    datetime64[us]
phone                        string
dtype: object

Missing values:
registration_date      0
phone                  0
email                374
dtype: int64


So we are retaining missing emails and phone values because we don't need to discard customer records simply because their contact information is incomplete.

Our primary objective is to understand and predict purchasing behaviour, not conducting customer communications.

Missing contact information does not prevent us from calculating historical spending, purchase frequency or other behavioural features.

## Final Customer Dimension Verification

In [7]:
print("CUSTOMER DIMENSION SUMMARY")
print("--------------------------")

print("Total records:", len(customers_clean))
print("Unique customer IDs:", customers_clean["customer_id"].nunique())
print("Duplicate customer IDs:", customers_clean["customer_id"].duplicated().sum())

print("\nData types:")
print(customers_clean.dtypes)

print("\nMissing values:")
print(customers_clean.isna().sum())

CUSTOMER DIMENSION SUMMARY
--------------------------
Total records: 3000
Unique customer IDs: 3000
Duplicate customer IDs: 0

Data types:
customer_id                     str
full_name                       str
gender                          str
birth_date           datetime64[us]
phone                        string
email                           str
city                            str
region                          str
loyalty_tier                    str
registration_date    datetime64[us]
dtype: object

Missing values:
customer_id            0
full_name              0
gender                 0
birth_date           560
phone                  0
email                374
city                   0
region                 0
loyalty_tier           0
registration_date      0
dtype: int64


Let's save the cleaned dataset

In [8]:
customers_clean.to_csv("../01_Data/processed/customers_clean.csv")

# Orders Table Cleaning and Preparation

This table will provide the foundation for our customer purchasing behaviour analysis and predictive targets.

We want to start by consolidating the two source tables:

- orders_2022_2023
- orders_2024

The quality check showed us that the consolidated table should contain 12,000 unique orders.

In [11]:
# Loading order tables separately for consolidation
orders_2022_2023 = load_csv("tbl_orders_2022_2023.csv")
orders_2024 = load_csv("tbl_orders_2024.csv")

# Appending orders_2024 to orders_2022_2023
orders = pd.concat(
    [orders_2022_2023, orders_2024],
    ignore_index = True
)

orders_clean = orders.copy()
print("Total order records:", len(orders_clean))
print("Distinct order IDs:", orders_clean["order_id"].nunique())


Total order records: 12000
Distinct order IDs: 12000


## Standardizing Payment Methods

From the data inventory notebook, we identified leading spaces in some payment methods.

For example:

- "Cash"
- "  Cash"

These should represent the same payment method.

In [12]:
# Remove leading and trailing whitespace
orders_clean["payment_method"] = (
    orders_clean["payment_method"].str.strip()
)

# Examine the standardized categories
print(orders_clean["payment_method"].value_counts())

payment_method
Cash             2070
Vodafone Cash    1990
InstaPay         1990
Credit Card      1986
Debit Card       1984
Fawry            1980
Name: count, dtype: int64


## Converting order dates

Why these transformations are important

1. Standardized payment methods prevent artificial categories during analysis.

2. Proper datetime values allow us to calculate purchase recency, frequency, historical spending and future 90-day outcomes.

In [13]:
orders_clean["order_date"] = pd.to_datetime(
    orders_clean["order_date"],
    errors="coerce"
)

print("Order date data type:", orders_clean["order_date"].dtype)

print(
    "Missing/invalid order dates:",
    orders_clean["order_date"].isna().sum()
)

Order date data type: datetime64[us]
Missing/invalid order dates: 0
